# Figure 1 — population cohort (Fleischer et al., GSE113957)
Everything computed from the population-cohort RNA-seq (Jana's tximport counts; 133 healthy donors aged 1–96 yo, the 10
HGPS donors excluded) → `data/`:
- differential expression of older (61–96 yo, Group4 + Group5) vs younger (1–13 yo, Group1) donors and preranked GSEA on
  the 60-pathway panel (Supplementary Figure 1Sd; x-axis of 1Sc);
- the per-donor interferon-γ response score (Figure 1d).

Run first (needs pydeseq2 and anndata); then `fibroblast_analysis.ipynb`.

In [1]:
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

sys.path.insert(0, "..")                                # shared modules
import gsea                                             # preranked GSEA on the DESeq2 Wald statistic

META = "../../metadata"
DATA = "data"
H5AD = "/data/land/3d-inter-paper-data/jana-aging/rna_counts/pyximport_processed.h5ad"
DEPOSITED = "/data/land/3d-inter-paper-data/jana-aging/deseq-young-vs-old-results"
GMT = f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.gmt"    # the 60-pathway panel as gene sets (read only)
pathways = json.load(open(f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.json"))

## Cohort counts
Protein-coding gene counts per donor (rounded, summed per gene name) and donor metadata.

In [2]:
a = ad.read_h5ad(H5AD)
X = a.X.toarray() if hasattr(a.X, "toarray") else a.X
pc = (a.var.gene_type == "protein_coding").values                       # deposited results are protein-coding genes
counts = pd.DataFrame(np.rint(X[:, pc]).astype(int), index=a.obs["Run"].values, columns=a.var.gene_name.values[pc])
counts = counts.T.groupby(level=0).sum().T                               # one column per gene name
meta = a.obs.set_index("Run")[["Age group", "disease", "Age", "sex"]].rename(columns={"Age group": "group"})
print(counts.shape[0], "donors;", counts.shape[1], "protein-coding genes;", (meta.disease == "Normal").sum(), "healthy")

143 donors; 19602 protein-coding genes; 133 healthy


## Differential expression and GSEA (Supplementary Figure 1Sd; x-axis of Supplementary Figure 1Sc)
DESeq2 of older vs younger healthy donors (genes with ≥ 10 counts in ≥ 5 donors), then preranked GSEA on the Wald
statistic (shared `../gsea.py`). The processing is first checked by rerunning Group4 vs Group1 against Jana's deposited
result. Writes `data/cohort_60plus_vs_young_deseq.csv` and `data/cohort_60plus_vs_young_gsea.csv`.

In [3]:
def deseq(old_groups, label):
    """DESeq2 of the given old groups (pooled) vs Group1; healthy donors; genes with >=10 counts in >=5 samples."""
    keep = meta[(meta.disease == "Normal") & meta.group.isin(["Group1", *old_groups])].index
    cnt = counts.loc[keep]
    cnt = cnt.loc[:, (cnt >= 10).sum(axis=0) >= 5]
    m = pd.DataFrame({"age": np.where(meta.loc[keep, "group"] == "Group1", "young", label)}, index=keep)
    dds = DeseqDataSet(counts=cnt, metadata=m, design_factors="age", quiet=True)
    dds.deseq2()
    st = DeseqStats(dds, contrast=["age", label, "young"], quiet=True)
    st.summary()
    n_old = (m.age == label).sum()
    n_young = (m.age == "young").sum()
    print(f"{label} vs young: {n_old} old, {n_young} young donors; {cnt.shape[1]} genes tested; "
          f"padj<0.05: {(st.results_df.padj < 0.05).sum()}")
    return st.results_df

In [4]:
# processing check: rerun Group4 vs Group1 and compare with Jana's deposited result
g4 = deseq(["Group4"], "Group4")
dep = pd.read_csv(f"{DEPOSITED}/Group4_vs_Group1_results.csv", index_col=0)
common = g4.index.intersection(dep.index)
r = np.corrcoef(g4.loc[common, "stat"].fillna(0), dep.loc[common, "stat"].fillna(0))[0, 1]
print(f"Group4 vs Group1 rerun vs deposited: {len(common)} shared genes, Wald stat r = {r:.4f}")

Fitting dispersions...
... done in 0.19 seconds.

Fitting MAP dispersions...


... done in 0.20 seconds.

Fitting LFCs...


... done in 0.22 seconds.



Group4 vs young: 34 old, 19 young donors; 13859 genes tested; padj<0.05: 3295
Group4 vs Group1 rerun vs deposited: 13859 shared genes, Wald stat r = 0.9871


In [5]:
# older (Group4 + Group5, 61-96 yo) vs younger (Group1, 1-13 yo)
old = deseq(["Group4", "Group5"], "old60plus")
old.to_csv(f"{DATA}/cohort_60plus_vs_young_deseq.csv")
res = gsea.prerank(gsea.deseq_rank(old), GMT)
res.to_csv(f"{DATA}/cohort_60plus_vs_young_gsea.csv", index=False)
print(f"GSEA: {(res['FDR q-val'] < 0.05).sum()} of {len(res)} pathways at FDR < 0.05")

Fitting dispersions...
... done in 0.16 seconds.

Fitting MAP dispersions...


... done in 0.18 seconds.

Fitting LFCs...
... done in 0.16 seconds.



old60plus vs young: 55 old, 19 young donors; 14029 genes tested; padj<0.05: 6345


GSEA: 28 of 60 pathways at FDR < 0.05


## Interferon-γ response score per donor (Figure 1d)
Per healthy donor: log2(CPM + 1) of expressed protein-coding genes (≥ 10 counts in ≥ 5 donors), each gene z-scored
across donors, averaged over the Hallmark interferon-γ response genes. Writes `data/cohort_isg_scores.csv`; the
correlation with age is computed in `fibroblast_analysis.ipynb`.

In [6]:
healthy = meta.index[meta.disease == "Normal"]
cnt = counts.loc[healthy]
cnt = cnt.loc[:, (cnt >= 10).sum(axis=0) >= 5]                          # expressed genes
logcpm = np.log2(cnt.div(cnt.sum(axis=1), axis=0) * 1e6 + 1)
z = (logcpm - logcpm.mean()) / logcpm.std()                              # z-score each gene across donors

ifng_genes = [g for g in set(pathways["Interferon gamma response (H)"]) if g in z.columns]
scores = pd.DataFrame({"IFN-γ response": z[ifng_genes].mean(axis=1)}).join(meta.loc[healthy, ["Age", "sex"]])
scores.index.name = "Run"
scores.to_csv(f"{DATA}/cohort_isg_scores.csv")
print(f"{len(scores)} healthy donors, ages {scores.Age.min()}-{scores.Age.max()}; "
      f"{len(ifng_genes)} expressed interferon-γ response genes")

133 healthy donors, ages 1-96; 168 expressed interferon-γ response genes
